In [9]:
import pandas as pd
import inspect
import importlib
import re
import os

In [10]:
# ================================
# Função para parsear docstrings estilo NumPy
# ================================
def parse_doc(doc: str):
    sections = {
        "summary": "",
        "parameters": "",
        "attributes": "",
        "see_also": "",
        "examples": "",
        "other": ""
    }
    if not doc:
        return sections
    pattern = re.compile(r"^(Parameters|Attributes|See Also|Examples)\n[-]+\n", re.M)
    parts = pattern.split(doc)
    sections["summary"] = parts[0].strip()
    for i in range(1, len(parts), 2):
        sec = parts[i].lower().replace(" ", "_")
        content = parts[i+1].strip() if i+1 < len(parts) else ""
        if sec in sections:
            sections[sec] = content
        else:
            sections["other"] += f"\n\n{parts[i]}\n{content}"
    return sections

In [8]:
# ================================
# Função 1: inspeciona o path do sklearn
# ================================
def inspect_sklearn_path(path: str) -> pd.DataFrame:
    """
    Recebe um path do sklearn (classe, função ou módulo)
    e retorna um dataframe com todas as informações possíveis
    """
    try:
        obj = importlib.import_module(path)
    except ModuleNotFoundError:
        # tenta importar como classe/função
        module_name, _, attr_name = path.rpartition(".")
        mod = importlib.import_module(module_name)
        obj = getattr(mod, attr_name)

    # identifica tipo
    if inspect.isclass(obj):
        obj_type = "class"
    elif inspect.isfunction(obj):
        obj_type = "function"
    elif inspect.ismodule(obj):
        obj_type = "module"
    else:
        obj_type = type(obj).__name__

    name = getattr(obj, "__name__", str(obj))
    full_name = path
    doc = inspect.getdoc(obj) or ""
    try:
        sig = str(inspect.signature(obj)) if callable(obj) else ""
    except (ValueError, TypeError):
        sig = ""

    parsed = parse_doc(doc)

    # atributos (somente para classes)
    attrs = []
    if inspect.isclass(obj):
        for attr_name, attr_val in inspect.getmembers(obj):
            if attr_name.startswith("_") and attr_name not in ["_estimator_type"]:
                continue
            # armazenar tipo, representação e doc
            try:
                attr_doc = inspect.getdoc(attr_val) or ""
            except:
                attr_doc = ""
            attrs.append(f"{attr_name}: {type(attr_val).__name__}, doc: {attr_doc[:30]}...")

        estimator_type = getattr(obj, "_estimator_type", "")
    else:
        estimator_type = ""

    # cria registro
    record = pd.DataFrame([{
        "full_name": full_name,
        "name": name,
        "type": obj_type,
        "signature": sig,
        "estimator_type": estimator_type,
        "summary": parsed["summary"],
        "parameters_doc": parsed["parameters"],
        "attributes_doc": parsed["attributes"],
        "attributes_list": "\n".join(attrs),
        "see_also": parsed["see_also"],
        "examples": parsed["examples"],
        "other": parsed["other"]
    }])
    return record

In [5]:
# ================================
# Função 2: gerencia o dataframe / TSV
# ================================
def manage_dataframe(df_path: str, record: pd.DataFrame = None) -> pd.DataFrame:
    """
    Abre ou cria o dataframe TSV. Se record for passado, atualiza e salva.
    """
    if os.path.exists(df_path):
        df = pd.read_csv(df_path, sep="\t", encoding="utf-8")
    else:
        # cria DataFrame vazio com colunas iguais às do record (se houver)
        if record is not None:
            df = pd.DataFrame(columns=record.columns)
        else:
            df = pd.DataFrame()

    if record is not None:
        if "full_name" in df.columns:
            # evita duplicatas pelo full_name
            df = pd.concat([df[df.full_name != record.full_name.iloc[0]], record], ignore_index=True)
        else:
            # DataFrame vazio sem coluna full_name
            df = pd.concat([df, record], ignore_index=True)

        # salva
        df.to_csv(df_path, sep="\t", index=False, encoding="utf-8")

    return df


In [6]:
# ================================
# Exemplo de uso
# ================================
paths = [
    "sklearn.naive_bayes.CategoricalNB",
    "sklearn.linear_model.LinearRegression",
    "sklearn.metrics.accuracy_score"
]

df_path = "sklearn_objects.tsv"

for path in paths:
    rec = inspect_sklearn_path(path)
    manage_dataframe(df_path, rec)

# abre dataframe final
df_final = manage_dataframe(df_path)
df_final.head()


,full_name,name,type,signature,estimator_type,summary,parameters_doc,attributes_doc,attributes_list,see_also,examples,other
0,sklearn.naive_bayes.CategoricalNB,CategoricalNB,class,"(*, alpha=1.0, force_alpha=True, fit_prior=Tru...",classifier,Naive Bayes classifier for categorical feature...,"alpha : float, default=1.0\n Additive (Lapl...",category_count_ : list of arrays of shape (n_f...,"_estimator_type: str, doc: str(object='') -> s...",BernoulliNB : Naive Bayes classifier for multi...,>>> import numpy as np\n>>> rng = np.random.Ra...,NaN
1,sklearn.linear_model.LinearRegression,LinearRegression,class,"(*, fit_intercept=True, copy_X=True, tol=1e-06...",regressor,Ordinary least squares Linear Regression.\n\nL...,"fit_intercept : bool, default=True\n Whethe...","coef_ : array of shape (n_features, ) or (n_ta...","_estimator_type: str, doc: str(object='') -> s...",Ridge : Ridge regression addresses some of the...,>>> import numpy as np\n>>> from sklearn.linea...,NaN
2,sklearn.metrics.accuracy_score,accuracy_score,function,"(y_true, y_pred, *, normalize=True, sample_wei...",NaN,Accuracy classification score.\n\nIn multilabe...,"y_true : 1d array-like, or label indicator arr...",NaN,NaN,balanced_accuracy_score : Compute the balanced...,>>> from sklearn.metrics import accuracy_score...,NaN


In [ ]:
import sklearn, inspect

inspect.signature(eval(df_final.full_name[0]))


<Signature (*, alpha=1.0, force_alpha=True, fit_prior=True, class_prior=None, min_categories=None)>

In [36]:
for full_name in df_final.full_name:
    name = full_name.split('.')[-1]
    object = eval(full_name)
    signature = str(inspect.signature(object)) if callable(object) else ""
    signature = inspect.signature(object)
    parameters = signature.parameters
    print(name.ljust(15), '\n\t', full_name, '\n\t', signature, '\n\t', parameters)

CategoricalNB   
	 sklearn.naive_bayes.CategoricalNB 
	 (*, alpha=1.0, force_alpha=True, fit_prior=True, class_prior=None, min_categories=None) 
	 OrderedDict({'alpha': <Parameter "alpha=1.0">, 'force_alpha': <Parameter "force_alpha=True">, 'fit_prior': <Parameter "fit_prior=True">, 'class_prior': <Parameter "class_prior=None">, 'min_categories': <Parameter "min_categories=None">})
LinearRegression 
	 sklearn.linear_model.LinearRegression 
	 (*, fit_intercept=True, copy_X=True, tol=1e-06, n_jobs=None, positive=False) 
	 OrderedDict({'fit_intercept': <Parameter "fit_intercept=True">, 'copy_X': <Parameter "copy_X=True">, 'tol': <Parameter "tol=1e-06">, 'n_jobs': <Parameter "n_jobs=None">, 'positive': <Parameter "positive=False">})
accuracy_score  
	 sklearn.metrics.accuracy_score 
	 (y_true, y_pred, *, normalize=True, sample_weight=None) 
	 OrderedDict({'y_true': <Parameter "y_true">, 'y_pred': <Parameter "y_pred">, 'normalize': <Parameter "normalize=True">, 'sample_weight': <Parameter "

In [42]:
list(parameters.keys())

['y_true', 'y_pred', 'normalize', 'sample_weight']

In [119]:
full_name = 'sklearn.linear_model.LinearRegression'
object = eval(full_name)
signature = str(inspect.signature(object)) if callable(object) else ""
signature = inspect.signature(object)
parameters = signature.parameters
print()
for param in list(parameters.keys()):
    print('=' * 100)
    print(param)  # 'fit_intercept'
    print(type(parameters[param]))  # <class 'inspect.Parameter'>
    print('=' * 100)
    print(parameters[param])  # <Parameter "normalize=True">
    print(parameters[param].name)  # 'normalize'
    print(parameters[param].default)  # True
    print(parameters[param].kind.name)  # KEYWORD_ONLY
    print(parameters[param].kind.value)  # 3
    print(parameters[param].KEYWORD_ONLY.value)  # 3
    print(parameters[param].POSITIONAL_ONLY)  # 0
    print(parameters[param].VAR_KEYWORD.name)  # VAR_KEYWORD
    print(parameters[param].VAR_KEYWORD.value)  # 4
    print(parameters[param].empty.__name__)  # inspect._empty
    print(parameters[param].empty)  # <class 'inspect._empty'>
    print(parameters[param].VAR_POSITIONAL)  # VAR_POSITIONAL
    print(parameters[param].annotation)  # <class 'inspect._empty'>



fit_intercept
<class 'inspect.Parameter'>
fit_intercept=True
fit_intercept
True
KEYWORD_ONLY
3
3
POSITIONAL_ONLY
VAR_KEYWORD
4
_empty
<class 'inspect._empty'>
VAR_POSITIONAL
<class 'inspect._empty'>
copy_X
<class 'inspect.Parameter'>
copy_X=True
copy_X
True
KEYWORD_ONLY
3
3
POSITIONAL_ONLY
VAR_KEYWORD
4
_empty
<class 'inspect._empty'>
VAR_POSITIONAL
<class 'inspect._empty'>
tol
<class 'inspect.Parameter'>
tol=1e-06
tol
1e-06
KEYWORD_ONLY
3
3
POSITIONAL_ONLY
VAR_KEYWORD
4
_empty
<class 'inspect._empty'>
VAR_POSITIONAL
<class 'inspect._empty'>
n_jobs
<class 'inspect.Parameter'>
n_jobs=None
n_jobs
None
KEYWORD_ONLY
3
3
POSITIONAL_ONLY
VAR_KEYWORD
4
_empty
<class 'inspect._empty'>
VAR_POSITIONAL
<class 'inspect._empty'>
positive
<class 'inspect.Parameter'>
positive=False
positive
False
KEYWORD_ONLY
3
3
POSITIONAL_ONLY
VAR_KEYWORD
4
_empty
<class 'inspect._empty'>
VAR_POSITIONAL
<class 'inspect._empty'>


In [ ]:
import inspect
import pandas as pd
import sklearn
import re

rows = []

def parse_doc(doc: str):
    """Divide a docstring numpy-style em blocos estruturados."""
    sections = {
        "summary": "",
        "parameters": "",
        "attributes": "",
        "see_also": "",
        "examples": "",
        "other": ""
    }
    if not doc:
        return sections

    # regex para capturar seções
    pattern = re.compile(r"^(Parameters|Attributes|See Also|Examples)\n[-]+\n", re.M)
    parts = pattern.split(doc)

    # primeira parte é o summary
    sections["summary"] = parts[0].strip()

    # percorrer pares (section_name, content)
    for i in range(1, len(parts), 2):
        sec = parts[i].lower().replace(" ", "_")
        content = parts[i+1].strip() if i+1 < len(parts) else ""
        if sec in sections:
            sections[sec] = content
        else:
            sections["other"] += f"\n\n{parts[i]}\n{content}"

    return sections

for method in [m for m in dir(sklearn.metrics) if not m.endswith("_")]:
    obj = getattr(sklearn.metrics, method)

    # tipo
    if isinstance(obj, type):
        obj_type = "class"
    elif callable(obj):
        obj_type = "function"
    else:
        obj_type = type(obj).__name__

    name = getattr(obj, "__name__", "")
    full_name = f"sklearn.metrics.{method}"
    doc = inspect.getdoc(obj) or ""
    try:
        sig = str(inspect.signature(obj)) if callable(obj) else ""
    except (ValueError, TypeError):
        sig = ""

    # dividir docstring
    parsed = parse_doc(doc)

    rows.append({
        "method": method,
        "type": obj_type,
        "name": name,
        "full_name": full_name,
        "signature": sig,
        "summary": parsed["summary"],
        "parameters": parsed["parameters"],
        "attributes": parsed["attributes"],
        "see_also": parsed["see_also"],
        "examples": parsed["examples"],
        "other": parsed["other"]
    })

df = pd.DataFrame(rows)

# salvar em TSV
df.to_csv("sklearn_metrics.tsv", sep="\t", index=False)
df

,method,type,name,full_name,signature,summary,parameters,attributes,see_also,examples,other
0,ConfusionMatrixDisplay,class,ConfusionMatrixDisplay,sklearn.metrics.ConfusionMatrixDisplay,"(confusion_matrix, *, display_labels=None)",Confusion Matrix visualization.\n\nIt is recom...,confusion_matrix : ndarray of shape (n_classes...,im_ : matplotlib AxesImage\n Image represen...,confusion_matrix : Compute Confusion Matrix to...,>>> import matplotlib.pyplot as plt\n>>> from ...,
1,DetCurveDisplay,class,DetCurveDisplay,sklearn.metrics.DetCurveDisplay,"(*, fpr, fnr, estimator_name=None, pos_label=N...",Detection Error Tradeoff (DET) curve visualiza...,fpr : ndarray\n False positive rate.\n\nfnr...,line_ : matplotlib Artist\n DET Curve.\n\na...,det_curve : Compute error rates for different ...,>>> import matplotlib.pyplot as plt\n>>> from ...,
2,DistanceMetric,class,DistanceMetric,sklearn.metrics.DistanceMetric,,Uniform interface for fast distance metric fun...,,,,>>> from sklearn.metrics import DistanceMetric...,
3,PrecisionRecallDisplay,class,PrecisionRecallDisplay,sklearn.metrics.PrecisionRecallDisplay,"(precision, recall, *, average_precision=None,...",Precision Recall visualization.\n\nIt is recom...,precision : ndarray\n Precision values.\n\n...,line_ : matplotlib Artist\n Precision recal...,precision_recall_curve : Compute precision-rec...,>>> import matplotlib.pyplot as plt\n>>> from ...,
4,PredictionErrorDisplay,class,PredictionErrorDisplay,sklearn.metrics.PredictionErrorDisplay,"(*, y_true, y_pred)",Visualization of the prediction error of a reg...,"y_true : ndarray of shape (n_samples,)\n Tr...",line_ : matplotlib Artist\n Optimal line re...,PredictionErrorDisplay.from_estimator : Predic...,>>> import matplotlib.pyplot as plt\n>>> from ...,
...,...,...,...,...,...,...,...,...,...,...,...
88,silhouette_samples,function,silhouette_samples,sklearn.metrics.silhouette_samples,"(X, labels, *, metric='euclidean', **kwds)",Compute the Silhouette Coefficient for each sa...,"X : {array-like, sparse matrix} of shape (n_sa...",,,>>> from sklearn.metrics import silhouette_sam...,
89,silhouette_score,function,silhouette_score,sklearn.metrics.silhouette_score,"(X, labels, *, metric='euclidean', sample_size...",Compute the mean Silhouette Coefficient of all...,"X : {array-like, sparse matrix} of shape (n_sa...",,,>>> from sklearn.datasets import make_blobs\n>...,
90,top_k_accuracy_score,function,top_k_accuracy_score,sklearn.metrics.top_k_accuracy_score,"(y_true, y_score, *, k=2, normalize=True, samp...",Top-k Accuracy classification score.\n\nThis m...,"y_true : array-like of shape (n_samples,)\n ...",,accuracy_score : Compute the accuracy score. B...,>>> import numpy as np\n>>> from sklearn.metri...,
91,v_measure_score,function,v_measure_score,sklearn.metrics.v_measure_score,"(labels_true, labels_pred, *, beta=1.0)",V-measure cluster labeling given a ground trut...,"labels_true : array-like of shape (n_samples,)...",,homogeneity_score : Homogeneity metric of clus...,Perfect labelings are both homogeneous and com...,
